# Python と Google Earth Engine

> **原典:** [Tutorial 11: Python and Google Earth Engine — WUR GeoScripting](https://github.com/GeoScripting-WUR/Python-GEE/blob/main/index.Rmd)  
> 著者: WUR GeoScripting チーム
>
> コメント・説明を日本語に翻訳し、コードのバグを修正しています。

## このチュートリアルの実行方法

このチュートリアルは **Jupyter Notebook** での実行を推奨します。
コードはインタラクティブなウィジェット（地図）を使用しており、Notebook 内でインラインに表示されます。


## 学習目標

- contextily で Web タイル衛星画像を取得・表示できる
- Google Earth Engine（Python API）で Sentinel-2 トゥルーカラー画像を取得できる
- 2種類の画像を Matplotlib で並べて比較できる

## Google Earth Engine とは

[Google Earth](https://earth.google.com/web/) はご存知の方も多いでしょう。Google Earth の内部では、強力なクラウドベースの地理空間解析プラットフォーム「**Google Earth Engine（GEE）**」が動いています。GEE は Google アカウントがあれば誰でも利用でき、膨大な地理空間データと計算リソースを提供します。

GEE はもともと **JavaScript API** として開発・公開されました。「JavaScript」「API」という言葉を分解して理解しましょう。

### API とは

API（Application Programming Interface）とは、異なるソフトウェア同士が決まったルールで通信するための「共通言語」です。例えば、天気予報データを取得したり、GIS ツール（GDAL など）を Python や R から操作する際に使われます。

### JavaScript とは

JavaScript（JS）は Web 開発で広く使われるプログラミング言語で、**オブジェクト指向プログラミング（OOP）** を採用しています。OOP では、データ（プロパティ）と機能（メソッド）を「オブジェクト」にまとめ、再利用しやすい形で記述します。Python も OOP に対応しており、GEE の Python API を使うことで OOP の考え方を自然に学ぶことができます。

---

## Google Earth Engine

GEE は膨大な地理空間データを提供するクラウドプラットフォームです。衛星画像・気候データ・地形データなど多種多様なデータセットが含まれており、環境モニタリング・資源管理・農業・気候変動研究など幅広い分野で活用されています。

利用可能なデータセットは [GEE データカタログ](https://developers.google.com/earth-engine/datasets/catalog) で確認できます。

---

## 解析概要：東京・千代田区の衛星画像比較

このチュートリアルでは、**東京・千代田区の 6km × 6km エリア**を2種類の画像ソースで取得して並べて比較します。

| 画像ソース | ライブラリ | 解像度・特徴 |
|-----------|-----------|-------------|
| Esri World Imagery | contextily（Web タイル） | ズームレベル 16（約 2.4m/px）・最新の航空・衛星画像 |
| Sentinel-2 SR | GEE（メジアンコンポジット） | 20m/px・雲マスク処理済み |

解析対象期間: **2025-03-01 〜 2025-05-30**

### 必要なパッケージ 
- `uv sync`で既にuv環境にインストールされているはずです。
```bash
uv add earthengine-api geemap geopandas contextily pyproj japanize-matplotlib python-dotenv
```

### .env ファイルの準備

認証情報はコードに直接書かず `.env` ファイルで管理します。

```bash
# プロジェクトルートで実行
cp .env.sample .env
```

`.env` を開いて `GEE_PROJECT_ID` に自分のプロジェクト ID を入力してください。

```
GEE_PROJECT_ID=your-project-id   # ← ここを書き換える
```

In [2]:
import ee
import geemap
import geopandas as gpd
import japanize_matplotlib  # matplotlib で日本語フォントを有効化

## Google Cloud の設定

GEE を Python から使うには、**Google Cloud プロジェクト**を作成して Earth Engine API を有効にする必要があります。
以下の手順を順番に行ってください（初回のみ）。

> **参考（画像付きで見やすい）:** [Google Earth Engine を Python で使う — Zenn](https://zenn.dev/fusic/articles/d21ac63d8d3c69)  
> 迷ったときはこちらのブログのスクリーンショットを参照してください。

---

### ステップ 1：Google Cloud プロジェクトを作成する

1. [Google Cloud Console](https://console.cloud.google.com/) にアクセスし、Google アカウントでログインします
2. 画面上部のプロジェクト選択欄をクリックし、**「新しいプロジェクト」** を選択します
3. プロジェクト名を入力して（例: `gee-tutorial`）**「作成」** をクリックします
4. 作成したプロジェクトが選択されていることを確認します

---

### ステップ 2：Earth Engine API を有効にする

1. 左上のハンバーガーメニュー（☰）→ **「APIとサービス」** → **「ライブラリ」** を選択します
2. 検索欄に `Earth Engine` と入力します
3. **「Google Earth Engine API」** を選択し、**「有効にする」** をクリックします

---

### ステップ 3：Earth Engine Code Editor でプロジェクトを登録する

1. [Earth Engine Code Editor](https://code.earthengine.google.com/) にアクセスします
2. **「I'M AUTHORIZED FOR AN EXISTING CLOUD PROJECT」** を選択します
3. ステップ 1 で作成した Google Cloud プロジェクトを選択します
4. **「REGISTER PROJECT」** をクリックして登録します
5. 課金プランの選択画面が表示されたら **「Unpaid usage（非課金）」** を選択します
   （個人の学習・検証目的であれば無料枠で十分です）

> **クォータについて:** GEE には EECU（Earth Engine Compute Units）のクォータ制限があります。クォータに達した場合は処理を分割するか、日を改めて実行してください。通常の学習・演習レベルでは問題になりません。

---

### ステップ 4：プロジェクト ID を確認する

Google Cloud Console の上部またはダッシュボードに表示されている **プロジェクト ID**（例: `gee-tutorial-123456`）をメモしておいてください。

> プロジェクト ID はプロジェクト名とは別に自動生成される場合があります（末尾に数字が付くことがあります）。
> Console 上部の「プロジェクト情報」カードで確認できます。

In [ ]:
import os
from dotenv import load_dotenv

# プロジェクトルートの .env を読み込む
load_dotenv('../.env')

PROJECT_ID = os.environ.get('GEE_PROJECT_ID')

if not PROJECT_ID or PROJECT_ID == 'your-project-id':
    raise ValueError(
        '.env ファイルに GEE_PROJECT_ID が設定されていません。\n'
        '手順: cp .env.sample .env を実行して GEE_PROJECT_ID を入力してください。'
    )

print(f'プロジェクト ID: {PROJECT_ID}')

## 認証（Authentication）

Google Cloud プロジェクトの準備ができたら、Python から GEE を使うための認証を行います。

1. 下のセルを実行すると URL が表示されます
2. URL をクリックして Google アカウントにログインします
3. 「Google に確認されていません」という警告が表示されたら **「続行」** をクリックします
4. すべてのチェックボックスにチェックを入れて許可します
5. 表示された認証コードをコピーし、入力欄に貼り付けて Enter を押します

> **2回目以降:** 同じ環境では `ee.Authenticate()` は不要です。`ee.Initialize(project=PROJECT_ID)` だけ実行すれば OK です。

In [ ]:
ee.Authenticate()

In [11]:
# プロジェクト ID を指定して Earth Engine ライブラリを初期化する
# earthengine-api 0.1.370 以降は project 引数が必須です
ee.Initialize(project=PROJECT_ID)

エラーや警告が表示されなければ認証成功です。おめでとうございます！

## ドキュメントの読み方：JavaScript と Python の違い

GEE のドキュメントは JavaScript（JS）と Python（Google Colab）の両方で例が書かれています。Python 版の整備が進んでいるため、JS の例を Python に読み替える力があると便利です。

主な構文の違いを以下に示します。

**変数の定義**

```python
# Python
variablename = 'value'
```
```javascript
// JavaScript（var / let / const の3種類がある）
var variablename = 'value'
let variablename = 'value'
const variablename = 'value'
```

**関数への引数の渡し方**

```python
# Python：キーワード引数を使う
result = img.reduceRegions(
    collection=regionCol,
    reducer=reducer,
    scale=285,
)
```
```javascript
// JavaScript：辞書（オブジェクト）を渡す
var result = img.reduceRegions({
  collection: regionCol,
  reducer: reducer,
  scale: 285,
});
```

---

## 解析エリア：東京・千代田区（6km × 6km）

解析の中心として**東京・千代田区**（皇居・東京駅周辺）を選びます。
まず、中心座標から 6km × 6km のバウンディングボックスを計算します。

| 座標系 | 用途 |
|--------|------|
| WGS84（EPSG:4326） | GEE の `ee.Geometry` |
| Web Mercator（EPSG:3857） | contextily のタイル取得 |

In [ ]:
import math
import numpy as np
from pyproj import Transformer

# 千代田区の中心座標（皇居周辺）
CENTER_LAT = 35.6938
CENTER_LON = 139.7536
SIZE_KM    = 6.0
START_DATE = '2025-03-01'
END_DATE   = '2025-05-30'

# WGS84 バウンディングボックスを計算
# 緯度方向: 1° ≈ 111km
# 経度方向: 1° ≈ 111km × cos(緯度)
delta_lat = (SIZE_KM / 2) / 111.0
delta_lon = (SIZE_KM / 2) / (111.0 * math.cos(math.radians(CENTER_LAT)))

west  = CENTER_LON - delta_lon
east  = CENTER_LON + delta_lon
south = CENTER_LAT - delta_lat
north = CENTER_LAT + delta_lat

# WGS84 → Web Mercator（EPSG:3857）に変換（contextily 用）
tf = Transformer.from_crs('EPSG:4326', 'EPSG:3857', always_xy=True)
west_m,  south_m = tf.transform(west,  south)
east_m,  north_m = tf.transform(east,  north)

print(f'WGS84:        W={west:.4f}, S={south:.4f}, E={east:.4f}, N={north:.4f}')
print(f'Web Mercator: W={west_m:.0f}, S={south_m:.0f}, E={east_m:.0f}, N={north_m:.0f}')

## ① Web タイル衛星画像（contextily）

[contextily](https://contextily.readthedocs.io/) は OpenStreetMap・Stamen・Esri など Web マップタイルを Python から取得するライブラリです。

ここでは **Esri World Imagery**（高解像度の航空・衛星画像）を使います。
タイルは Web Mercator（EPSG:3857）座標系で返されます。

- `zoom=16` → 約 2.4m/pixel
- 返り値 `img` は RGBA の numpy 配列 `(H, W, 4)`
- 返り値 `ext` は `(west, east, south, north)` の Web Mercator 座標

In [ ]:
import contextily as ctx
import os

os.makedirs('../output', exist_ok=True)

# Esri World Imagery タイルをダウンロードして GeoTIFF に保存
img_tile, ext_tile = ctx.bounds2raster(
    west_m, south_m, east_m, north_m,
    path='../output/chiyoda_tile.tif',
    source=ctx.providers.Esri.WorldImagery,
    zoom=16,
)

# contextily は RGBA (4チャンネル) を返すため RGB のみ抽出
rgb_tile = img_tile[:, :, :3]

print(f'Web タイル: shape={img_tile.shape}, extent={[round(v) for v in ext_tile]}')

## ② Sentinel-2 トゥルーカラー（GEE）

GEE の [COPERNICUS/S2_SR_HARMONIZED](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED) コレクションから Sentinel-2 Surface Reflectance 画像を取得します。

処理の流れ:

1. **filterBounds / filterDate:** 対象エリア・期間に絞り込む
2. **CLOUDY_PIXEL_PERCENTAGE フィルター:** 雲量 20% 超の画像を除外
3. **SCL バンドによる雲マスク:** 雲・雲影ピクセルを除去
4. **median():** 残ったピクセルの中央値（メジアン）合成
5. **clip():** 解析エリアで切り取る

使用バンド: **B4（赤）/ B3（緑）/ B2（青）** → トゥルーカラー RGB

> GEE のコードはこの時点では計算されません。API コール（命令書）を定義しているだけで、実際の計算はデータが必要になったときにクラウド上で行われます。

In [ ]:
# 雲マスク関数（SCL バンドを使用）
def mask_s2_clouds(image):
    scl = image.select('SCL')
    # 除外クラス: 3=雲影, 8=中確度雲, 9=高確度雲, 10=Cirrus 雲
    mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return image.updateMask(mask)

# 解析エリアを GEE Geometry として定義
roi = ee.Geometry.Rectangle([west, south, east, north])

# 画像コレクションを取得・フィルタリング
s2_collection = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    .map(mask_s2_clouds)
)

# このタイミングで GEE クラウドへの API コールが実行される
print(f'条件を満たす画像数: {s2_collection.size().getInfo()} 枚')

# RGB バンドの中央値コンポジット → エリアでクリップ
s2_median = (
    s2_collection
    .select(['B4', 'B3', 'B2'])
    .median()
    .clip(roi)
)

## ③ GEE 画像を numpy 配列に変換する

`geemap.ee_to_numpy()` を使って GEE の画像を numpy 配列に変換します。
内部では `sampleRectangle()` が呼ばれるため、取得できる画素数に上限があります。

6km エリアを 10m 解像度で取得すると 600 × 600 = 360,000 px/バンドとなり上限を超える可能性があるため、
`reproject()` で **20m 解像度**（300 × 300 px/バンド）に変換してから取得します。

Sentinel-2 SR の輝度値は反射率 × 10000 の整数（0〜10000 程度）です。
表示用に **3000 でクリップして 0〜1 に正規化**します（都市部のアスファルト・建物が適切な明るさになります）。

In [ ]:
# 20m 解像度に変換（sampleRectangle の上限を回避: 300×300 px/バンド）
s2_reproj = s2_median.reproject(crs='EPSG:4326', scale=20)

# GEE → numpy 配列（H, W, 3）
arr_s2 = geemap.ee_to_numpy(s2_reproj, bands=['B4', 'B3', 'B2'], region=roi)
print(f'Sentinel-2 配列: shape={arr_s2.shape}, dtype={arr_s2.dtype}')

# 輝度値（0-10000）を 0-1 に正規化（3000 でクリップ）
arr_s2_norm = np.clip(arr_s2.astype('float32') / 3000.0, 0, 1)

## ④ 2つの画像を並べて比較する

Matplotlib の `subplots(1, 2)` で左右に並べて表示します。

- **左:** contextily（Web タイル）— 高解像度・最新の商用画像
- **右:** Sentinel-2（GEE）— 10〜20m 解像度・オープンデータ・雲マスク済みメジアン合成

contextily 画像は Web Mercator 座標（`extent=ext_tile`）で表示します。
Sentinel-2 は `ee_to_numpy` で取得した配列をそのまま表示します（ピクセル座標軸）。

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(15, 7))

# 左：contextily Web タイル
axes[0].imshow(rgb_tile, extent=ext_tile)
axes[0].set_title(
    'Web タイル衛星画像\n（Esri World Imagery, zoom=16, ~2.4m/px）',
    fontsize=12
)
axes[0].axis('off')

# 右：Sentinel-2 トゥルーカラー
axes[1].imshow(arr_s2_norm)
axes[1].set_title(
    f'Sentinel-2 トゥルーカラー\n（{START_DATE} 〜 {END_DATE} 中央値合成, 20m/px）',
    fontsize=12
)
axes[1].axis('off')

fig.suptitle('東京・千代田区 6km × 6km', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../output/chiyoda_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('比較画像を ../output/chiyoda_comparison.png に保存しました')

---

## まとめ

このチュートリアルでは以下を学びました。

| テーマ | 内容 |
|--------|------|
| OOP | クラス・インスタンス・プロパティ・メソッド・継承 |
| GEE 基礎 | API の概念・認証・Google Cloud プロジェクト設定 |
| contextily | Web タイル（Esri World Imagery）のダウンロードと表示 |
| Sentinel-2 | 雲マスク・メジアンコンポジット・トゥルーカラー RGB |
| 座標変換 | WGS84 ↔ Web Mercator（pyproj） |
| データ変換 | GEE Image → numpy 配列（reproject + ee_to_numpy） |
| 可視化 | Matplotlib での2画像並べて比較 |

GEE のコードはローカルの計算機ではなく **Earth Engine のクラウド**上で実行されます。そのため、ローカル PC のスペックに関わらず大規模な地理空間解析を高速に行えます。

さらに学習したい場合は以下のリソースを参照してください。

- [Google Earth Engine ドキュメント](https://developers.google.com/earth-engine)
- [Sentinel-2 SR データカタログ（GEE）](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED)
- [contextily ドキュメント](https://contextily.readthedocs.io/)
- [geemap サンプル Notebook 集](https://github.com/gee-community/geemap/tree/master/examples/notebooks)